# Süt ineklerinde davranış anomalisi tespiti (MMCOWS)

**Amaç:** Etiketi olmayan sensör verisinden, bir ineğin davranışındaki anormal değişimi **saatlik** veriyle yakalayan bir sistem kurmak ve ne kadar işe yaradığını dürüstçe ölçmek.

**Veri:** [MMCOWS](https://github.com/neis-lab/mmcows) (NeurIPS 2024). 10 inek, 14 gün, Wisconsin'de bir ahır. Lisans: makale CC BY-NC 4.0 diyor; ticari kullanım öncesi Hugging Face veri kartını kontrol et.

**Akış:**
1. Kurulum ve veri indirme
2. Veriyi tanıma (dosyalar, sağlık kayıtları, C↔T eşleşmesi, vücut sıcaklığı)
3. Saatlik özellik tablosu (inek × saat)
4. Skor tasarımı: neden mutlak skor değil, yönlü "durgunluk" skoru
5. Son pipeline: filtre → sürüden fark → kişisel robust z → durgunluk → yumuşatma → alarm
6. Eşiği alarm bütçesine göre seçmek
7. Sentetik anomali testi (tatbikat) ve pencere karşılaştırması
8. Veri boşlukları (sağım saatleri)
9. Sonuçlar ve sınırlılıklar

## 1. Kurulum ve veri indirme

In [1]:
import collections
import glob
import os
from collections import defaultdict

import numpy as np
import pandas as pd
from google.colab import drive
from huggingface_hub import HfApi, hf_hub_download

drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/cow-anomaly/data"
BASE = f"{DATA}/sensor_data"
REPO = "neis-lab/mmcows"
os.makedirs(DATA, exist_ok=True)
!pip -q install remotezip

Mounted at /content/drive


Veri seti herkese açık, Hugging Face girişi gerekmiyor. Token gerekirse koda yazma, Colab'ın sol menüsündeki **Secrets** (anahtar simgesi) bölümüne `HF_TOKEN` adıyla ekleyip `google.colab.userdata.get("HF_TOKEN")` ile oku.

Önce indirmeden dosya listesine bakıyoruz: veri setinde terabaytlarca kamera verisi var.

In [2]:
for f in HfApi().list_repo_tree(REPO, repo_type="dataset", recursive=True):
    size = getattr(f, "size", None)
    if size and size > 1e9:
        continue
    print(f.path, f"{size / 1e6:.1f} MB" if size else "")

15s_interval_combined_videos 
1s_interval_images 
1s_interval_images_3hr 
uwb_synced_images 
.gitattributes 0.0 MB
15s_interval_combined_videos/0804.mp4 718.0 MB
1s_interval_images/README.md 0.0 MB
1s_interval_images/proj_mat.zip 0.1 MB
1s_interval_images_3hr/Description.txt 0.0 MB
1s_interval_images_3hr/proj_mat.zip 0.1 MB
README.md 0.0 MB
behavior_labels.zip 32.1 MB
pred_labels.zip 20.0 MB
sensor_model_weights.zip 370.4 MB
trained_models.zip 929.5 MB
uwb_synced_images/proj_mat.zip 0.1 MB


In [3]:
for name in ["README.md", "behavior_labels.zip", "pred_labels.zip"]:
    hf_hub_download(REPO, name, repo_type="dataset", local_dir=DATA)

Sensör verisi tek bir 18 GB'lık zip. `remotezip` ile tamamını indirmeden içini listeliyoruz.

In [4]:
from remotezip import RemoteZip

URL = f"https://huggingface.co/datasets/{REPO}/resolve/main/sensor_data.zip"
sizes = defaultdict(lambda: [0, 0])

with RemoteZip(URL) as z:
    for info in z.infolist():
        key = "/".join(info.filename.split("/")[:3])
        sizes[key][0] += 1
        sizes[key][1] += info.file_size

for key, (n, b) in sorted(sizes.items()):
    print(f"{key:50s} {n:5d} files {b / 1e6:10.1f} MB")

sensor_data/                                           1 files        0.0 MB
sensor_data/behavior_labels/                           1 files        0.0 MB
sensor_data/behavior_labels/individual                17 files       32.1 MB
sensor_data/main_data/                                 1 files        0.0 MB
sensor_data/main_data/ankle                          161 files        7.1 MB
sensor_data/main_data/cbt                             11 files        3.9 MB
sensor_data/main_data/immu                           193 files     7267.3 MB
sensor_data/main_data/milk                            17 files        0.0 MB
sensor_data/main_data/pressure                       193 files     4339.5 MB
sensor_data/main_data/thi                              8 files        4.7 MB
sensor_data/main_data/uwb                            193 files       29.7 MB
sensor_data/main_data/weather                         18 files        9.1 MB
sensor_data/sub_data/                                  1 files        0.0 MB

Yalnızca küçük sensör klasörlerini çıkarıyoruz (~60 MB). `immu`, `pressure` ve `head_direction` GB'larca, şimdilik gerek yok.

In [5]:
KEEP = (
    "sensor_data/main_data/ankle", "sensor_data/main_data/cbt",
    "sensor_data/main_data/thi", "sensor_data/main_data/milk",
    "sensor_data/main_data/uwb", "sensor_data/main_data/weather",
    "sensor_data/sub_data/health_records",
)

with RemoteZip(URL) as z:
    members = [i for i in z.infolist() if i.filename.startswith(KEEP) and not i.is_dir()]
    for info in members:
        # Skipping existing files lets us resume after a Colab disconnect.
        if not os.path.exists(f"{DATA}/{info.filename}"):
            z.extract(info, DATA)

print(len(members), "files ready")

396 files ready


## 2. Veriyi tanıma

| Klasör | Kim | Sıklık | Ne |
|---|---|---|---|
| `cbt` | C01–C10 | 1 dk | Çekirdek vücut sıcaklığı |
| `thi` | S01–S07 ortam sensörü | 1 dk | Ahır sıcaklık, nem, THI |
| `ankle` | C01–C10, günlük dosya | 1 dk | Yatıyor (1) / ayakta (0) |
| `uwb` | T01–T10 boyun etiketi (+T13, T14 sabit referans) | 15 sn | Boyun konumu x, y, z (cm) |
| `milk` | 16 inek | günde 1 | Süt verimi, laktasyon günü |
| `health_records` | 16 inek | olay | Çiftlik kayıtları |
| `weather` | ahır dışı | 5 dk | xlsx, kullanmıyoruz |

Zaman damgaları Unix saniyesi (UTC); yerel saat `America/Chicago`.

In [6]:
folders = ["main_data/cbt", "main_data/thi", "main_data/ankle",
           "main_data/uwb", "main_data/milk", "sub_data/health_records"]

for folder in folders:
    path = sorted(glob.glob(f"{BASE}/{folder}/**/*.csv", recursive=True))[0]
    # utf-8-sig strips the BOM that health_records files start with.
    df = pd.read_csv(path, encoding="utf-8-sig")
    print(f"\n== {folder} ({os.path.basename(path)}) shape={df.shape}")
    print(df.head(3).to_string())


== main_data/cbt (C01.csv) shape=(19831, 2)
      timestamp  temperature_C
0  1.689961e+09          38.35
1  1.689961e+09          38.35
2  1.689961e+09          38.35

== main_data/thi (S01.csv) shape=(19831, 4)
      timestamp  temperature_C  humidity_per     THI
0  1.689961e+09         25.744        51.772  73.156
1  1.689961e+09         25.453        51.764  72.770
2  1.689961e+09         25.186        52.330  72.475

== main_data/ankle (C01_0721.csv) shape=(690, 3)
      timestamp             datetime  lying
0  1.689961e+09  2023-07-21 12:30:00      1
1  1.689961e+09  2023-07-21 12:31:00      1
2  1.689961e+09  2023-07-21 12:32:00      1

== main_data/uwb (T01_0721.csv) shape=(2754, 4)
      timestamp  coord_x_cm  coord_y_cm  coord_z_cm
0  1.689961e+09       119.2        49.6       117.8
1  1.689961e+09       136.8        40.2       115.7
2  1.689961e+09       146.9        46.8       119.0

== main_data/milk (C01.csv) shape=(15, 3)
      timestamp  milk_weight_kg    DIM
0  1.6899

### Sağlık kayıtları: kayıt penceresinde hastalık etiketi var mı?
Kayıtlar bütün laktasyon geçmişini kapsıyor; bizim penceremiz 21 Temmuz – 4 Ağustos 2023.

In [7]:
files = sorted(glob.glob(f"{BASE}/sub_data/health_records/*.csv"))
health = pd.concat(
    [pd.read_csv(f, encoding="utf-8-sig").assign(cow=os.path.basename(f)[:-4]) for f in files]
)
health["time"] = pd.to_datetime(health["Date"], format="%Y-%m-%d %I %p", errors="coerce")
print("unparsed dates:", health["time"].isna().sum())

window = health[(health["time"] >= "2023-07-21") & (health["time"] < "2023-08-05")]
print(len(window), "events inside the recording window")
print(window.groupby("Event").size().sort_values(ascending=False))
print(window[["cow", "time", "Event", "Specific event", "Description"]]
      .sort_values(["cow", "time"]).to_string())

unparsed dates: 0
22 events inside the recording window
Event
Other                 12
Preg Check             7
Bred                   1
Enroll in protocol     1
DNB                    1
dtype: int64
    cow                time               Event      Specific event                                                             Description
5   C01 2023-07-21 00:00:00               Other               Onexp                                                           onexp - 683CC
43  C02 2023-07-21 00:00:00               Other               Onexp                                                           onexp - 683CC
47  C03 2023-07-21 00:00:00               Other               Onexp                                                           onexp - 683CC
46  C03 2023-07-31 00:00:00          Preg Check          Preg Check                                                     Preg - M - 116 DAYS
29  C04 2023-07-21 00:00:00               Other               Onexp                                 

**Sonuç:** Pencerede hastalık, tedavi ya da kızgınlık kaydı yok. `Onexp` = C01–C10'un deneye alınması. C15'teki `Bred` senkronizasyon protokolü (Ovsynch) ve C15'te sensör yok. **Hazır anomali etiketi yok**, değerlendirmeyi vekil sinyal ve sentetik testle yapacağız. Kızgınlık tespiti bu veriyle test edilemez.

### C (inek) ↔ T (boyun etiketi) eşleşmesi
README eşleşmeyi vermiyor. Veriden doğruluyoruz: ayak sensörü "yatıyor" derken aynı ineğin boyun etiketi belirgin alçalmalı.

In [8]:
DAY = "0722"  # a full day, unlike 0721 which starts at 12:30

lying = {}
for c in range(1, 11):
    df = pd.read_csv(f"{BASE}/main_data/ankle/C{c:02d}/C{c:02d}_{DAY}.csv")
    ts = pd.to_datetime(df["timestamp"], unit="s").dt.floor("min")
    s = pd.Series(df["lying"].values, index=ts)
    lying[f"C{c:02d}"] = s[~s.index.duplicated()]

neck_z = {}
for t in range(1, 11):
    df = pd.read_csv(f"{BASE}/main_data/uwb/T{t:02d}/T{t:02d}_{DAY}.csv")
    ts = pd.to_datetime(df["timestamp"], unit="s").dt.floor("min")
    # UWB samples every 15 s, so the minute median matches the ankle sensor's resolution.
    neck_z[f"T{t:02d}"] = df.groupby(ts)["coord_z_cm"].median()

pair_gap = pd.DataFrame(index=list(lying), columns=list(neck_z), dtype=float)
for c, l in lying.items():
    for t, z in neck_z.items():
        j = pd.concat([l, z], axis=1, keys=["lying", "z"]).dropna()
        pair_gap.loc[c, t] = j.loc[j["lying"] == 1, "z"].mean() - j.loc[j["lying"] == 0, "z"].mean()

print(pair_gap.round(0))
print(pair_gap.idxmin(axis=1))

      T01   T02   T03   T04   T05   T06   T07   T08   T09   T10
C01 -32.0  -8.0  -1.0   1.0  -7.0 -10.0  -4.0  -0.0   1.0   8.0
C02 -16.0 -40.0  -6.0 -16.0 -11.0  -9.0  -3.0  -6.0  -3.0  11.0
C03  -1.0  -2.0 -25.0   3.0   5.0   4.0  -6.0   4.0   2.0  -4.0
C04  -2.0 -18.0   5.0 -38.0 -15.0  -6.0   1.0  -1.0  -7.0   3.0
C05 -10.0 -19.0   6.0 -20.0 -31.0  -9.0  -0.0  -9.0 -10.0  10.0
C06  -9.0 -14.0  -1.0  -7.0  -8.0 -34.0  -2.0  -8.0 -15.0  -0.0
C07  -2.0   2.0  -8.0   7.0   5.0  -5.0 -26.0   2.0   4.0   2.0
C08  -0.0  -5.0  -1.0   2.0  -3.0  -9.0  -3.0 -28.0  -9.0 -12.0
C09   4.0  -8.0  -2.0 -10.0 -10.0 -20.0   2.0 -11.0 -26.0 -10.0
C10   8.0   7.0  -8.0   7.0   9.0  -4.0   2.0  -5.0  -5.0 -42.0
C01    T01
C02    T02
C03    T03
C04    T04
C05    T05
C06    T06
C07    T07
C08    T08
C09    T09
C10    T10
dtype: object


**Sonuç:** Her satırda en negatif değer köşegende → C01↔T01 … C10↔T10.

### Çekirdek vücut sıcaklığı: temizleme ve saatlik özet

In [9]:
cbt = pd.concat(
    [pd.read_csv(f).assign(cow=os.path.basename(f)[:-4])
     for f in sorted(glob.glob(f"{BASE}/main_data/cbt/*.csv"))]
)
cbt["time"] = pd.to_datetime(cbt["timestamp"], unit="s", utc=True).dt.tz_convert("America/Chicago")

# Readings like 21.9 C mean the logger was outside the body.
bad = (cbt["temperature_C"] < 36) | (cbt["temperature_C"] > 41)
print("implausible readings removed:", int(bad.sum()))
cbt = cbt[~bad]

hourly_cbt = (
    cbt.set_index("time").groupby("cow")["temperature_C"]
    .resample("1h").mean().rename("cbt_mean").reset_index()
)
print(hourly_cbt.groupby("cow")["cbt_mean"]
      .agg(mean="mean", max="max", hours_above_39=lambda s: (s > 39.0).sum()).round(2))

implausible readings removed: 8
      mean    max  hours_above_39
cow                              
C01  38.47  39.23               2
C02  38.57  39.25              15
C03  38.46  39.18               2
C04  38.47  39.25               8
C05  38.42  39.19               7
C06  38.59  39.13               7
C07  38.43  39.13               7
C08  38.48  39.15               6
C09  38.58  39.36              26
C10  38.40  39.18               7


## 3. Saatlik özellik tablosu (inek × saat)

Saatlik, çünkü gerçek tasma saatte bir paket gönderiyor: üretimdeki kısıtı simüle ediyoruz.

| Sütun | Ne | Rolü |
|---|---|---|
| `lying_frac` | Saatin yatarak geçen oranı | Model girdisi |
| `move_cm` | Saatlik yürünen mesafe (UWB) | Model girdisi |
| `neck_z`, `neck_z_std` | Boyun yüksekliği ve oynaması | Model girdisi |
| `ankle_minutes`, `uwb_samples` | Ölçüm sayısı (tam saat: 60 ve 240) | Kalite filtresi |
| `thi` | Ahır sıcaklık-nem indeksi | Bağlam |
| `cbt_mean` | Çekirdek vücut sıcaklığı | **Sadece doğrulama**, modele girmez |

In [10]:
def to_local_hour(ts):
    return pd.to_datetime(ts, unit="s", utc=True).dt.tz_convert("America/Chicago").dt.floor("h")

rows = []
for i in range(1, 11):
    cow, tag = f"C{i:02d}", f"T{i:02d}"

    ankle = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{BASE}/main_data/ankle/{cow}/*.csv")))
    ankle["hour"] = to_local_hour(ankle["timestamp"])
    a = ankle.groupby("hour")["lying"].agg(lying_frac="mean", ankle_minutes="count")

    uwb = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{BASE}/main_data/uwb/{tag}/*.csv")))
    uwb = uwb.sort_values("timestamp")
    uwb["hour"] = to_local_hour(uwb["timestamp"])
    step = np.hypot(uwb["coord_x_cm"].diff(), uwb["coord_y_cm"].diff())
    # Steps across a time gap or a position glitch would inflate movement, so drop them.
    uwb["step_cm"] = step.where((uwb["timestamp"].diff() <= 30) & (step < 1000))
    u = uwb.groupby("hour").agg(
        move_cm=("step_cm", "sum"), neck_z=("coord_z_cm", "mean"),
        neck_z_std=("coord_z_cm", "std"), uwb_samples=("coord_z_cm", "count"),
    )

    rows.append(a.join(u, how="outer").assign(cow=cow))

feat = pd.concat(rows)
feat.index.name = "hour"
feat = feat.reset_index()

thi = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{BASE}/main_data/thi/*.csv")))
thi["hour"] = to_local_hour(thi["timestamp"])
thi_h = thi.groupby("hour")["THI"].mean().rename("thi").reset_index()

hourly = (
    feat.merge(thi_h, on="hour", how="left")
        .merge(hourly_cbt.rename(columns={"time": "hour"}), on=["cow", "hour"], how="left")
)
hourly.to_parquet(f"{DATA}/hourly.parquet")

print(hourly.shape)
print(hourly.isna().mean().round(3).to_string())
print(hourly.describe().T.round(2).to_string())

(3313, 10)
hour             0.000
lying_frac       0.000
ankle_minutes    0.000
move_cm          0.003
neck_z           0.003
neck_z_std       0.003
uwb_samples      0.003
cow              0.000
thi              0.000
cbt_mean         0.000
                count     mean      std      min      25%      50%       75%       max
lying_frac     3313.0     0.55     0.37     0.00     0.22     0.57      0.93      1.00
ankle_minutes  3313.0    59.74     3.59     1.00    60.00    60.00     60.00     60.00
move_cm        3303.0  8617.94  4901.76  1476.66  4826.26  7523.49  11358.18  36943.19
neck_z         3303.0   101.96    14.85    61.22    90.43   100.40    111.93    156.78
neck_z_std     3303.0    28.16    16.68     4.85    17.08    25.24     33.08    156.44
uwb_samples    3303.0   226.94    34.88    55.00   238.00   239.00    239.00    243.00
thi            3313.0    71.58     5.24    58.73    68.32    71.77     75.45     82.60
cbt_mean       3313.0    38.49     0.24    37.76    38.32    38

## 4–5. Pipeline

Bu bölümden sonrası yalnızca `hourly.parquet` dosyasını kullanır; Colab yeniden başlarsa 1. bölümdeki ilk hücreyi çalıştırıp doğrudan buradan devam edebilirsin.

Adımlar:
1. **Kalite filtresi:** eksik ölçümlü saatleri at.
2. **Log:** `move_cm` sağa çarpık, birkaç aşırı saat ölçeği domine etmesin.
3. **Sürüden fark:** her saatte sürü medyanını çıkar. Sıcak hava, sağım ve yem saatleri gibi herkesi etkileyen şeyler temizlenir.
4. **Kişisel robust z-score:** her ineğin farkını kendi medyanı ve MAD'ine göre ölçekle. Hareketli ve sakin inekler aynı ölçeğe gelir.
5. **Durgunluk skoru:** çok yatma + az hareket + baş aşağıda. Yönlü, mutlak değer değil.
6. **Yumuşatma:** saatlik skor çok zıplıyor, son `window` saatin ortalaması.
7. **Alarm:** yumuşatılmış skor eşiği **art arda en az 3 saat** aşarsa.

In [11]:
FEATURES = ["lying_frac", "move_cm", "neck_z", "neck_z_std"]
Z_COLS = [f"{f}_z" for f in FEATURES]
TZ = "America/Chicago"


def robust_z(values, groups):
    g = values.groupby(groups)
    mad = g.transform(lambda s: (s - s.median()).abs().median())
    # Floor stops near-constant cows from producing infinite z-scores.
    return (values - g.transform("median")) / (1.4826 * mad).clip(lower=1e-6)


def mark_alarms(df, col, threshold, min_hours=3):
    d = df.sort_values(["cow", "hour"]).copy()
    d["high"] = d[col] > threshold
    gap = d.groupby("cow")["hour"].diff() != pd.Timedelta("1h")
    # A new run starts when the flag flips, the cow changes or hours are missing.
    new_run = (d["high"] != d.groupby("cow")["high"].shift()) | gap
    d["run"] = new_run.cumsum()
    d["run_len"] = d.groupby("run")["high"].transform("size")
    return d["high"] & (d["run_len"] >= min_hours)


def count_events(df):
    d = df.sort_values(["cow", "hour"])
    prev = d.groupby("cow")["alarm"].shift(fill_value=False).astype(bool)
    gap = d.groupby("cow")["hour"].diff() != pd.Timedelta("1h")
    return int((d["alarm"] & (~prev | gap)).sum())


def score_pipeline(raw, thr=None, window=6, q=0.95):
    df = raw[(raw["ankle_minutes"] >= 45) & (raw["uwb_samples"] >= 180)].copy()
    df["move_cm"] = np.log1p(df["move_cm"])

    for f in FEATURES:
        dev = df[f] - df.groupby("hour")[f].transform("median")
        df[f"{f}_z"] = robust_z(dev, df["cow"])

    df["lethargy"] = (df["lying_frac_z"] - df["move_cm_z"] - df["neck_z_z"]) / 3
    df = df.sort_values(["cow", "hour"])
    df["leth_smooth"] = df.groupby("cow")["lethargy"].transform(
        lambda s: s.rolling(window, min_periods=window // 2 + 1).mean()
    )

    if thr is None:
        thr = df["leth_smooth"].quantile(q)
    df["alarm"] = mark_alarms(df, "leth_smooth", thr)
    return df, thr


raw = pd.read_parquet(f"{DATA}/hourly.parquet")
N_COWS = raw["cow"].nunique()
WEEKS = (raw["hour"].max() - raw["hour"].min()) / pd.Timedelta("7D")
print(raw.shape, "cows:", N_COWS, "weeks:", round(WEEKS, 2))

(3313, 10) cows: 10 weeks: 1.97


## 4. Skor tasarımı: neden yönlü skor?

İlk denemede 4 özelliğin **mutlak** z-skorlarının ortalamasını aldık. Bu skor vekil sinyalle (sürüye göre çekirdek sıcaklık, `cbt_z`) ilişkisiz çıktı. Sebep: mutlak değer "çok yatmak" ile "hiç yatmamak"ı aynı sayıyor; sinyal ise yönlü.

`cbt_z`, sıcaklığın da aynı dönüşümden (sürüden fark + kişisel z) geçmiş hâli. Ham sıcaklıkla kıyaslamak yanlış olurdu: ham sıcaklığı en çok herkesi aynı anda etkileyen sıcak hava yükseltiyor.

In [12]:
scored, _ = score_pipeline(raw, thr=np.inf)
cbt_dev = scored["cbt_mean"] - scored.groupby("hour")["cbt_mean"].transform("median")
scored["cbt_z"] = robust_z(cbt_dev, scored["cow"])
scored["abs_score"] = scored[Z_COLS].abs().mean(axis=1)

for col in ["abs_score", "lethargy"]:
    deciles = pd.qcut(scored[col], 10, labels=False)
    print(f"\n{col}: mean cbt_z by decile")
    print(scored.groupby(deciles)["cbt_z"].mean().round(3).to_string())

print("\nSpearman correlation with cbt_z:")
print(scored[Z_COLS + ["cbt_z"]].corr(method="spearman")["cbt_z"].round(3).to_string())


abs_score: mean cbt_z by decile
abs_score
0   -0.032
1   -0.004
2    0.017
3   -0.064
4    0.072
5    0.020
6    0.062
7   -0.044
8    0.046
9   -0.001

lethargy: mean cbt_z by decile
lethargy
0   -0.391
1   -0.272
2   -0.289
3   -0.154
4    0.025
5   -0.004
6    0.156
7    0.192
8    0.446
9    0.364

Spearman correlation with cbt_z:
lying_frac_z    0.237
move_cm_z      -0.170
neck_z_z       -0.246
neck_z_std_z   -0.111
cbt_z           1.000


Durgunluk–sıcaklık ilişkisi her inekte ayrı ayrı var mı, yoksa birkaç inek mi sürüklüyor?

In [13]:
per_cow = pd.Series({
    cow: g["lethargy"].corr(g["cbt_z"], method="spearman")
    for cow, g in scored.groupby("cow")
}).round(3)
print(per_cow.to_string())
print("positive cows:", int((per_cow > 0).sum()), "/", N_COWS)

C01    0.225
C02    0.349
C03    0.212
C04    0.179
C05    0.160
C06    0.479
C07    0.155
C08    0.139
C09    0.253
C10    0.267
positive cows: 10 / 10


**Dikkat:** Yönü `cbt` ilişkisine bakarak seçtik, bu yüzden `cbt` artık tamamen bağımsız bir test değil. Yön, alan bilgisiyle (hastalık davranışı = durgunluk) de uyumlu. Ayrıca yatmak vücut sıcaklığını kendisi artırabilir: bu ilişki hastalık kanıtı değil, tutarlılık kanıtı. Asıl test 7. bölümdeki sentetik tatbikat.

Yumuşatma neden gerekli: yüksek skorlu saatler dağınık, nadiren art arda geliyor.

In [14]:
high = scored["lethargy"] > scored["lethargy"].quantile(0.90)
next_high = high.groupby(scored["cow"]).shift(-1, fill_value=False).astype(bool)
print("high hours:", int(high.sum()), "| followed by another high hour:", int((high & next_high).sum()))

high hours: 302 | followed by another high hour: 26


## 6. Eşiği alarm bütçesine göre seçmek

Eşik keyfi değil, iş gereksiniminden: çiftlik sahibi **inek başına haftada ~1 olaya** bakabilir (10 inekte günde 1–2 bildirim). Olay = art arda alarm saatlerinden oluşan blok.

In [15]:
def threshold_for_budget(raw, window, target=1.0):
    base, _ = score_pipeline(raw, thr=np.inf, window=window)
    best = None
    for q in np.arange(0.85, 0.99, 0.005):
        t = base["leth_smooth"].quantile(q)
        rate = count_events(base.assign(alarm=mark_alarms(base, "leth_smooth", t))) / N_COWS / WEEKS
        if best is None or abs(rate - target) < abs(best["rate"] - target):
            best = {"window": window, "q": round(float(q), 3), "thr": float(t), "rate": float(rate)}
    return best


budget6 = threshold_for_budget(raw, window=6)
print(budget6)

{'window': 6, 'q': 0.935, 'thr': 0.46626000147628627, 'rate': 1.0151057401812689}


## 7. Sentetik anomali testi (tatbikat)

Gerçek hastalık etiketi olmadığı için "kontrollü duman" çıkarıyoruz: bir ineği 24 saatliğine yapay olarak durgun gösteriyoruz (yatma +0,3, hareket ×0,6) ve sistem fark ediyor mu bakıyoruz.

- **Eşik sabit:** gerçek sistemde eşik bozuk veri gelince kendini ayarlamaz.
- **Kirli pencereler:** tatbikat zaten doğal bir alarmın üstüne denk gelirse "yakaladı" saymak sonucu şişirir. Bunları ayrı raporluyoruz.

In [16]:
DRILL_DAYS = ["2023-07-24", "2023-07-27", "2023-07-30", "2023-08-02"]
DRILL_HOURS = ["08:00", "20:00"]


def drill(raw, cow, start, thr, window=6, lying_add=0.3, move_mult=0.6):
    end = start + pd.Timedelta("24h")
    fake = raw.copy()
    mask = (fake["cow"] == cow) & (fake["hour"] >= start) & (fake["hour"] < end)
    fake.loc[mask, "lying_frac"] = (fake.loc[mask, "lying_frac"] + lying_add).clip(upper=1)
    fake.loc[mask, "move_cm"] = fake.loc[mask, "move_cm"] * move_mult

    res, _ = score_pipeline(fake, thr=thr, window=window)
    w = res[(res["cow"] == cow) & (res["hour"] >= start) & (res["hour"] < end)]

    first = w.loc[w["alarm"], "hour"].min()
    detected = pd.notna(first)
    delay_h = (first - start) / pd.Timedelta("1h") if detected else np.nan
    return detected, delay_h


def had_alarm(df, cow, start):
    end = start + pd.Timedelta("24h")
    w = df[(df["cow"] == cow) & (df["hour"] >= start) & (df["hour"] < end)]
    return bool(w["alarm"].any())


def run_drills(raw, thr, window=6):
    clean, _ = score_pipeline(raw, thr=thr, window=window)
    rows = []
    for cow in sorted(raw["cow"].unique()):
        for day in DRILL_DAYS:
            for hh in DRILL_HOURS:
                start = pd.Timestamp(f"{day} {hh}", tz=TZ)
                detected, delay_h = drill(raw, cow, start, thr, window)
                rows.append({"cow": cow, "start": start, "detected": detected,
                             "delay_h": delay_h, "clean_alarm": had_alarm(clean, cow, start)})
    return pd.DataFrame(rows)


def summarize(drills):
    ok = drills[~drills["clean_alarm"]]
    return {"drills": len(drills), "contaminated": round(float(drills["clean_alarm"].mean()), 2),
            "detection_rate": round(float(ok["detected"].mean()), 2),
            "median_delay_h": float(ok["delay_h"].median())}


drills6 = run_drills(raw, budget6["thr"], window=6)
print(summarize(drills6))

{'drills': 80, 'contaminated': 0.16, 'detection_rate': 0.84, 'median_delay_h': 9.0}


### Pencere karşılaştırması: yanlış alarm ↔ gecikme dengesi

Her pencere **kendi** eşiğiyle (aynı bütçe kuralı) karşılaştırılıyor; 6 saatlik eşiği 3 saatlik skora uygulamak adil olmaz. Her pencere 80 tatbikat demek, birkaç dakika sürer.

In [17]:
summary = []
for window in [3, 6, 12]:
    budget = budget6 if window == 6 else threshold_for_budget(raw, window=window)
    drills = drills6 if window == 6 else run_drills(raw, budget["thr"], window=window)
    summary.append({**budget, **summarize(drills)})

summary = pd.DataFrame(summary)
summary["thr"] = summary["thr"].round(3)
summary["rate"] = summary["rate"].round(2)
print(summary.to_string(index=False))

 window     q   thr  rate  drills  contaminated  detection_rate  median_delay_h
      3 0.905 0.620  0.96      80          0.16            0.70             8.0
      6 0.935 0.466  1.02      80          0.16            0.84             9.0
     12 0.950 0.342  1.02      80          0.24            0.84            10.0


## 8. Veri boşlukları: sağım saatleri

In [18]:
missing = raw[(raw["ankle_minutes"] < 45) | (raw["uwb_samples"] < 180)]
print(missing["hour"].dt.hour.value_counts().sort_index().to_string())

hour
2       1
4     140
7      10
12      9
15    133
16      3
21      1


Eksik saatler her gün **04:00** ve **15:00**'te yığılıyor (10 inek × 14 gün ≈ 140). İnekler sağımhaneye gidip sensör ağının menzilinden çıkıyor: sistem günde iki saat kör.

## 9. Sonuçlar ve sınırlılıklar

- **C↔T eşleşmesi** veriden doğrulandı: C01↔T01 … C10↔T10.
- **Mutlak skor** vekil sinyalle (sürüye göre çekirdek sıcaklık) ilişkisiz. **Yönlü durgunluk skoru** ise düzenli artıyor: en düşük onluk dilimde `cbt_z` −0,39, en yüksek dilimlerde +0,36/+0,45. İlişki 10 ineğin 10'unda pozitif (Spearman 0,14–0,48).
- **Yumuşatma gerekli:** yüksek skorlu 302 saatin yalnızca 26'sını yine yüksek bir saat izliyor.
- **Eşik:** 6 saatlik skorun %93,5 dilimi (0,466) → inek başına haftada 1,02 olay.
- **Pencere karşılaştırması** (her pencere kendi bütçe eşiğiyle, 80'er tatbikat):

| Pencere | Haftalık olay | Kirli tatbikat | Yakalama (temiz) | Ortanca gecikme |
|---|---|---|---|---|
| 3 saat | 0,96 | %16 | %70 | 8 saat |
| **6 saat** | 1,02 | %16 | **%84** | 9 saat |
| 12 saat | 1,02 | %24 | %84 | 10 saat |

6 saat en iyi denge: 3 saat daha gürültülü olduğu için aynı bütçede daha az yakalıyor; 12 saat aynı oranı daha geç yakalıyor ve alarmları uzadığı için doğal alarmlarla daha çok çakışıyor.

- **Sağım saatlerinde** (04:00, 15:00) veri yok: sistem günde iki saat kör.

**Sınırlılıklar**
- 10 inek, 14 gün; gerçek hastalık ya da kızgınlık etiketi yok.
- `cbt` vekil sinyal; yön seçiminde kullanıldığı için tam bağımsız değil, ve yatmak sıcaklığı kendisi artırabilir.
- Sentetik bozma yalnızca yatma ve hareketi değiştiriyor; gerçek durgunlukta boyun yüksekliği de düşer. Gece zaten yatan inekte +0,3 etkisiz (1'de kesiliyor).
- `move_cm` UWB titremesini içeriyor (hareketsiz saatte bile ~15 m).

**Sonraki adımlar:** Isolation Forest ile karşılaştırma, farklı bozma şiddetleri, `move_cm` için konum yumuşatma, kendi çiftlik verisine geçiş.